In [ ]:
import os
import numpy as np
import pandas as pd


import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


PHREEQC_CSV = "Nitrate_Benchmark_4.csv"
UQ_CSV      = "Benchmark4_generalization.csv"  
OUT_DIR     = "generalization_from_csv"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "plots"), exist_ok=True)

output_column = "N(5)(mol/kgw)"

CANDIDATE_CONST_COLS = ['porosity', 'velocity', 'dispersivity', 'k_Fe', 'k_C', 'C_Fe3', 'K_Fe']


def add_sim_id(df: pd.DataFrame) -> pd.DataFrame:
    const_cols = [c for c in CANDIDATE_CONST_COLS if c in df.columns]
    if not const_cols:
        raise ValueError("No invariant candidate columns found in PHREEQC_CSV. "
                         "Update CANDIDATE_CONST_COLS to match the data.")

    const_block = df[const_cols].copy()
    for c in const_cols:
        if pd.api.types.is_numeric_dtype(const_block[c]):
            const_block[c] = const_block[c].round(8)

    sim_key = const_block.astype(str).agg('|'.join, axis=1)
    sim_id, _ = pd.factorize(sim_key, sort=True)

    out = df.copy()
    out["__sim_id__"] = sim_id
    return out


def sim_level_table(df_phr: pd.DataFrame) -> pd.DataFrame:
    if "__sim_id__" not in df_phr.columns:
        raise ValueError("df_phr must contain __sim_id__.")

    g = df_phr.groupby("__sim_id__", sort=True)

    x_min = g["dist_x"].min()
    x_max = g["dist_x"].max()
    L = (x_max - x_min).replace(0.0, np.nan)

    sim_params = g[[c for c in CANDIDATE_CONST_COLS if c in df_phr.columns]].first()

    tmp = df_phr.copy()
    tmp["_xmin"] = tmp.groupby("__sim_id__", sort=False)["dist_x"].transform("min")
    inlet = tmp[tmp["dist_x"].values == tmp["_xmin"].values].copy()
    if "C_NO3" not in inlet.columns:
        raise ValueError("PHREEQC_CSV missing required column 'C_NO3'.")
    C_NO3_in_med = inlet.groupby("__sim_id__", sort=True)["C_NO3"].median()

    out = sim_params.copy()
    out["L"] = L
    out["C_NO3_in_med"] = C_NO3_in_med

    if "C_DOC" in df_phr.columns:
        out["C_DOC_med"] = g["C_DOC"].median()

    if "velocity" in out.columns:
        v = out["velocity"].astype(float).replace(0.0, np.nan)
        out["t_adv"] = out["L"].astype(float) / v
    else:
        out["t_adv"] = np.nan

    cols_for_score = ["velocity", "dispersivity", "porosity", "k_C", "k_Fe", "C_Fe3", "K_Fe"]
    cols_for_score = [c for c in cols_for_score if c in out.columns]

    if len(cols_for_score) == 0:
        raise ValueError("No columns available for OOD scoring. Check PHREEQC_CSV columns.")

    score_parts = []
    for c in cols_for_score:
        x = out[c].astype(float)
        med = float(np.nanmedian(x.values))
        mad = float(np.nanmedian(np.abs(x.values - med))) + 1e-12
        rz = np.abs((x.values - med) / mad)
        out[f"rz_{c}"] = rz
        score_parts.append(rz.reshape(-1, 1))

    score_mat = np.hstack(score_parts) 
    out["OOD_score_max_rz"] = np.nanmax(score_mat, axis=1)
    out["OOD_score_mean_rz"] = np.nanmean(score_mat, axis=1)

    return out.reset_index()


def attach_uq_and_metrics(df_uq: pd.DataFrame) -> pd.DataFrame:
    need_cols = ["__sim_id__", output_column, "PINN_ens_mean", "PI_lo", "PI_hi"]
    missing = [c for c in need_cols if c not in df_uq.columns]
    if missing:
        raise ValueError(f"UQ CSV missing required columns: {missing}\n"
                         f"Expected at least: {need_cols}")

    df = df_uq.copy()
    for c in [output_column, "PINN_ens_mean", "PI_lo", "PI_hi"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["abs_err"] = np.abs(df["PINN_ens_mean"] - df[output_column])
    df["covered"] = ((df[output_column] >= df["PI_lo"]) & (df[output_column] <= df["PI_hi"])).astype(float)
    df["pi_width"] = df["PI_hi"] - df["PI_lo"]
    return df


def sim_aggregate(df_point: pd.DataFrame) -> pd.DataFrame:
    def _rmse(yhat, y):
        e = (yhat - y).to_numpy(dtype=float)
        return float(np.sqrt(np.nanmean(e * e)))

    g = df_point.groupby("__sim_id__", sort=True)

    rows = []
    for sid, d in g:
        rows.append({
            "__sim_id__": sid,
            "n_points": int(len(d)),
            "RMSE_sim": _rmse(d["PINN_ens_mean"], d[output_column]),
            "MAE_sim": float(np.nanmean(d["abs_err"].to_numpy(dtype=float))),
            "Coverage_sim": float(np.nanmean(d["covered"].to_numpy(dtype=float))),
            "PI_width_sim": float(np.nanmean(d["pi_width"].to_numpy(dtype=float))),
        })
    return pd.DataFrame(rows).sort_values("__sim_id__").reset_index(drop=True)


def make_id_ood_labels(sim_tbl: pd.DataFrame, quant=0.10) -> tuple[pd.DataFrame, float]:
    if "OOD_score_max_rz" not in sim_tbl.columns:
        raise ValueError("sim_tbl must contain 'OOD_score_max_rz'.")

    thr = float(sim_tbl["OOD_score_max_rz"].quantile(1.0 - quant))
    out = sim_tbl.copy()
    out["is_OOD"] = (out["OOD_score_max_rz"] >= thr).astype(int)
    out["split_label"] = np.where(out["is_OOD"] == 1, f"OOD(top{int(quant*100)}%)", f"ID(bottom{int((1-quant)*100)}%)")
    return out, thr


def summarize_id_ood(sim_join: pd.DataFrame) -> pd.DataFrame:
    def _summ(d: pd.DataFrame) -> pd.Series:
        return pd.Series({
            "n_sims": int(len(d)),
            "RMSE_mean": float(d["RMSE_sim"].mean()),
            "RMSE_median": float(d["RMSE_sim"].median()),
            "Coverage_mean": float(d["Coverage_sim"].mean()),
            "PI_width_mean": float(d["PI_width_sim"].mean()),
        })

    return sim_join.groupby("split_label", sort=True).apply(_summ).reset_index()


def plot_scatter(sim_join: pd.DataFrame, xcol: str, ycol: str, fname: str, ylabel=None):
    plt.figure(figsize=(6.2, 4.5))
    colors = sim_join["is_OOD"].map({0: "#1f77b4", 1: "#d62728"}).values
    plt.scatter(sim_join[xcol], sim_join[ycol], s=28, c=colors, alpha=0.85, edgecolors="none")
    plt.xlabel(xcol)
    plt.ylabel(ylabel or ycol)
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(fname, dpi=180)
    plt.close()

def main():
    df_phr = pd.read_csv(PHREEQC_CSV)
    df_uq  = pd.read_csv(UQ_CSV)

    if df_phr.shape[1] < 6:
        raise ValueError("PHREEQC_CSV has < 6 columns, cannot apply df.iloc[:,5] != -99 filter safely.")
    df_phr = df_phr[df_phr.iloc[:, 5] != -99].copy()

    df_phr = add_sim_id(df_phr)

    if "__sim_id__" not in df_uq.columns:
        raise ValueError(
            "UQ CSV must have '__sim_id__'.\n"
            "Regenerate UQ CSV from df_test.copy() that already contains '__sim_id__'."
        )

    sim_tbl = sim_level_table(df_phr)
    sim_tbl_fn = os.path.join(OUT_DIR, "generalization_sim_table.csv")
    sim_tbl.to_csv(sim_tbl_fn, index=False)

    df_uq_m = attach_uq_and_metrics(df_uq)
    sim_perf = sim_aggregate(df_uq_m)
    sim_perf_fn = os.path.join(OUT_DIR, "generalization_sim_perf_table.csv")
    sim_perf.to_csv(sim_perf_fn, index=False)

    sim_join = sim_tbl.merge(sim_perf, on="__sim_id__", how="inner")
    if sim_join.empty:
        raise ValueError(
            "Join sim_tbl and sim_perf is empty.\n"
            "Check that __sim_id__ in UQ_CSV matches the __sim_id__ computed from PHREEQC_CSV."
        )

    sim_join, thr = make_id_ood_labels(sim_join, quant=0.10)
    tab = summarize_id_ood(sim_join)
    tab["OOD_threshold_max_rz"] = thr
    tab_fn = os.path.join(OUT_DIR, "generalization_id_vs_ood_table.csv")
    tab.to_csv(tab_fn, index=False)

    plot_scatter(sim_join, "OOD_score_max_rz", "RMSE_sim",
                 os.path.join(OUT_DIR, "plots", "rmse_vs_oodscore.png"),
                 ylabel="RMSE per simulation")
    plot_scatter(sim_join, "OOD_score_max_rz", "Coverage_sim",
                 os.path.join(OUT_DIR, "plots", "coverage_vs_oodscore.png"),
                 ylabel="Coverage per simulation")
    plot_scatter(sim_join, "OOD_score_max_rz", "PI_width_sim",
                 os.path.join(OUT_DIR, "plots", "piwidth_vs_oodscore.png"),
                 ylabel="Mean PI width per simulation")

    print("Wrote:")
    print(" -", sim_tbl_fn)
    print(" -", sim_perf_fn)
    print(" -", tab_fn)
    print(" -", os.path.join(OUT_DIR, "plots", "rmse_vs_oodscore.png"))
    print(" -", os.path.join(OUT_DIR, "plots", "coverage_vs_oodscore.png"))
    print(" -", os.path.join(OUT_DIR, "plots", "piwidth_vs_oodscore.png"))


if __name__ == "__main__":
    main()